In [1]:
import os
import glob
import pandas as pd

In [35]:
# Путь к папке, где находятся Excel файлы
folder_path = r"C:\Млн\Вторичка\Новая папка"

# Создаём пустой DataFrame для накопления данных
all_data = pd.DataFrame()

# Используем glob для поиска всех Excel файлов в папке
excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

# Проходим по каждому файлу и добавляем его данные в DataFrame
for file_path in excel_files:
    try:
        df = pd.read_excel(file_path)  # Читаем Excel файл в DataFrame
    except:
        print(file_path)
    df.columns = df.columns.str.capitalize()


    all_data = pd.concat([all_data, df], ignore_index=True)  # Добавляем данные в общий DataFrame

In [36]:
all_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 764642 entries, 0 to 764641
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Дата                 764642 non-null  datetime64[us]
 1   Локация              764642 non-null  str           
 2   Город                764642 non-null  str           
 3   Метро                469025 non-null  str           
 4   Тип помещения        764636 non-null  str           
 5   Отделка              764642 non-null  str           
 6   Кол-во комнат        764622 non-null  float64       
 7   Площадь, кв.м        764642 non-null  float64       
 8   Площадь кухни, кв.м  764642 non-null  float64       
 9   Жилая площадь, кв.м  764642 non-null  float64       
 10  Цена лота, руб.      764642 non-null  int64         
 11  Этаж                 764642 non-null  int64         
 12  Балконы/лоджии       764642 non-null  int64         
dtypes: datetime64[us](1), flo

In [37]:
all_data['Цена за м2'] = all_data['Цена лота, руб.'] / all_data['Площадь, кв.м']

In [40]:
all_data['Кол-во комнат'].unique()

array([1, 2, 3, 4, 5, 6, 0])

In [39]:
# Удаляем строки, где нет данных о комнатах
all_data = all_data.dropna(subset=['Кол-во комнат'])

# Преобразуем в целые числа
all_data['Кол-во комнат'] = all_data['Кол-во комнат'].astype(int)

In [41]:
all_data['Дата'].value_counts()

Дата
2026-01-30    116477
2025-12-29    114345
2026-02-28    112182
2026-04-30    111948
2026-03-30    106891
2026-05-30    104771
2026-06-30     98008
Name: count, dtype: int64

In [45]:
all_data['Дата'].value_counts()

Дата
2026-01-30    99002
2025-12-29    97203
2026-02-28    95341
2026-04-30    95159
2026-03-30    90852
2026-05-30    89045
2026-06-30    83314
Name: count, dtype: int64

In [43]:
# Границы по каждому городу и месяцу
q10 = (
    all_data.groupby([all_data['Город'], all_data['Дата'].dt.to_period('M')])['Цена за м2']
      .transform(lambda x: x.quantile(0.10))
)

q95 = (
    all_data.groupby([all_data['Город'], all_data['Дата'].dt.to_period('M')])['Цена за м2']
      .transform(lambda x: x.quantile(0.95))
)

# Удаляем выбросы
all_data = all_data[(all_data['Цена за м2'] >= q10) & (all_data['Цена за м2'] <= q95)]

In [44]:
all_data.drop(columns='Цена за м2', inplace=True)

In [46]:
all_data.to_csv('C:\Млн\Вторичка\Новая папка\Млн_вторичка12-06_без выбросов.csv', index=False)